In [0]:
%sql
-- STEP 1: Get previous encounter per patient
CREATE OR REPLACE TEMP VIEW encounters_with_previous AS
SELECT
    encounter_id,
    patient_id,
    start_date,
    stop_date AS end_date,

    DATE_TRUNC('month', start_date) AS encounter_month,

    LAG(stop_date) OVER (
        PARTITION BY patient_id 
        ORDER BY start_date, encounter_id
    ) AS previous_end_date,

    LAG(encounter_id) OVER (
        PARTITION BY patient_id 
        ORDER BY start_date, encounter_id
    ) AS previous_encounter_id

FROM gold.fact_encounters;


-- STEP 2: Classify eligible + readmission
CREATE OR REPLACE TEMP VIEW classified_encounters AS
SELECT
    encounter_id,
    patient_id,
    start_date,
    end_date,
    encounter_month,

    DATEDIFF(start_date, previous_end_date) AS days_since_previous,

    -- Eligible encounters
    CASE 
        WHEN previous_encounter_id IS NOT NULL
             AND start_date >= previous_end_date
        THEN 1 ELSE 0
    END AS is_eligible,

    -- Readmission within 30 days
    CASE 
        WHEN previous_encounter_id IS NOT NULL
             AND start_date >= previous_end_date
             AND DATEDIFF(start_date, previous_end_date) <= 30
        THEN 1 ELSE 0
    END AS is_readmission

FROM encounters_with_previous;


-- STEP 3: Final KPI table
CREATE OR REPLACE TABLE gold.kpi_6 AS
SELECT
    DATE_FORMAT(encounter_month, 'yyyy-MM') AS month,

    SUM(is_eligible) AS eligible_encounters,
    SUM(is_readmission) AS readmissions,

    ROUND(
        SUM(is_readmission) * 100.0 / SUM(is_eligible),
        2
    ) AS readmission_rate

FROM classified_encounters

--  Only valid encounters
WHERE is_eligible = 1

GROUP BY encounter_month
ORDER BY month;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_5 AS
SELECT
    payer_id,
    AVG(total_claim_cost) AS avg_claim_cost
FROM gold.fact_encounters
GROUP BY payer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_4 AS
SELECT *
FROM (
    SELECT
        c.year,
        c.half_year,
        p.procedure_code,
        AVG(p.base_cost) AS avg_cost,
        COUNT(*) AS procedure_count,
        ROW_NUMBER() OVER (
            PARTITION BY c.year, c.half_year
            ORDER BY AVG(p.base_cost) DESC
        ) AS rank
    FROM gold.fact_procedures p
    JOIN gold.dim_calendar c
        ON p.start_date = c.date   -- ✅ FIXED
    GROUP BY c.year, c.half_year, p.procedure_code
)
WHERE rank <= 10;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_3 AS
WITH base AS (
    SELECT
        c.year_month,
        f.payer_id,
        CASE 
            WHEN f.payer_coverage = 0 THEN 1 
            ELSE 0 
        END AS is_zero_coverage
    FROM gold.fact_encounters f
    JOIN gold.dim_calendar c
        ON f.start_date = c.date
)

SELECT
    year_month,
    payer_id,
    COUNT(*) AS total_encounters,
    SUM(is_zero_coverage) AS zero_coverage_count,
    ROUND(100.0 * SUM(is_zero_coverage) / COUNT(*), 2) AS zero_coverage_pct,
    ROUND(100.0 * (COUNT(*) - SUM(is_zero_coverage)) / COUNT(*), 2) AS covered_pct
FROM base
GROUP BY year_month, payer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_2 AS
SELECT
    c.year_month,
    CASE 
        WHEN datediff(f.stop_date, f.start_date) > 1 THEN 'Over 24 hrs'
        ELSE 'Under 24 hrs'
    END AS duration_type,
    COUNT(*) AS encounter_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY c.year_month),
        2
    ) AS percentage
FROM gold.fact_encounters f
JOIN gold.dim_calendar c
    ON f.start_date = c.date
GROUP BY c.year_month, duration_type;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_1 AS
SELECT
    c.year_quarter,
    f.encounter_class,
    COUNT(*) AS encounter_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY c.year_quarter),
        2
    ) AS percentage
FROM gold.fact_encounters f
JOIN gold.dim_calendar c
    ON f.start_date = c.date
GROUP BY c.year_quarter, f.encounter_class;

In [0]:
%sql
SELECT COUNT(*)
FROM gold.fact_encounters f
JOIN gold.dim_calendar c
ON f.start_date = c.date;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;